In [3]:
"""Small, relationship-rich test for atomiccim_bind.

Run after building the extension:
    python3 test_public_architecture.py

The two axes intentionally mean different things:
    HORIZONTAL: signal-processing order
    VERTICAL:   control/precision order
"""

import gc

import atomiccim_bind as ac


H, V = ac.Axis.HORIZONTAL, ac.Axis.VERTICAL
FIRST, NEXT = ac.Inheritance.FIRST_CHILD, ac.Inheritance.LINKED_CHILD
FF = ac.MacroColumnOfAPC.FEEDFORWARD_MESSAGE
ERROR = ac.MacroColumnOfAPC.ERROR_SLOT
WEIGHT = ac.MacroColumnOfAPC.WEIGHT_SLOT
F64 = ac.RegionDataType.FLOAT64_T


def schema():
    """Give three equal-sized regions different logical jobs."""
    layout = ac.LayoutSpanAndPercentageCarrier()
    layout.FeedForward = layout.ErrorSlot = layout.WeightSlot = 1

    dtype = ac.InitialRegionalDtypeConf()
    dtype.FEEDFORWARD_MESSAGE = dtype.ERROR_SLOT = dtype.WEIGHT_SLOT = F64

    protocol = ac.InitialRegionalProtocol()
    protocol.FEEDFORWARD_MESSAGE = ac.RegionProtocol.ATOMIC_WORD_ARRAY
    protocol.ERROR_SLOT = ac.RegionProtocol.ATOMIC_WORD_ARRAY
    protocol.WEIGHT_SLOT = ac.RegionProtocol.PRIVATE_REGION
    return layout, dtype, protocol


def expect(result, apc):
    """Assert a navigation result and return its APC."""
    status, found_apc = result.as_tuple()
    assert result and result.found
    assert result.MutationOP_ == result.status == status == ac.NavigationStatus.FOUND
    assert result.APCPtr_.GetThisSlotIdx() == found_apc.GetThisSlotIdx()
    assert found_apc.GetThisSlotIdx() == apc.GetThisSlotIdx()
    return found_apc


def main():
    layout, dtype, protocol = schema()
    fabric = ac.Fabric()
    assert fabric.InitializeFabricWithPtrTable(7, ac.MINIMUM_APC_CELL_COUNT)
    assert fabric.IsFabricActive() and fabric.is_active()

    # Exercise both creation forms: caller-provided APC and returned APC.
    model = ac.APC()
    assert fabric.CreateAPC(model, True, True, layout, dtype, protocol)

    def make(h_root=False, v_root=False):
        return fabric.create_apc(h_root, v_root, layout, dtype, protocol)

    alternate = make(True, True)
    sensor, predictor, error, precision, scratch = [make() for _ in range(5)]
    assert all(node.IsActiveAPC() for node in (
        model, alternate, sensor, predictor, error, precision, scratch
    ))
    assert len({node.GetThisSlotIdx() for node in (
        model, alternate, sensor, predictor, error, precision, scratch
    )}) == 7

    # H topology: model -> [sensor, predictor].
    assert sensor.AttachMeToAnother(model, H, FIRST)
    assert sensor.AttachSiblingOrChild(predictor, H, NEXT)
    expect(model.FindMyNext(H, FIRST), sensor)
    expect(sensor.FindMyNext(H, NEXT), predictor)
    expect(predictor.FindPrevious(H), sensor)

    # Exercise every mutation shape while preserving a meaningful final graph.
    assert model.DetachMyChild(sensor, H)             # model -> [predictor]
    assert sensor.AttachMeToAnother(predictor, H, NEXT)
    assert sensor.DetachMeFromAnotherEdge(H)
    assert sensor.AttachMeToAnother(predictor, H, NEXT)
    assert sensor.DetachAndReAttachMeToThisParent(alternate, H)
    assert sensor.DetachAndReattachMeAsEquivelentSibbling(predictor, H)
    expect(model.FindMyNext(H, FIRST), predictor)
    expect(predictor.FindMyNext(H, NEXT), sensor)
    assert sensor.FindMyNext(H, NEXT).status == ac.NavigationStatus.NONE

    # V topology is independent: model -> [precision, error].
    assert precision.AttachMeToAnother(model, V, FIRST)
    assert precision.AttachSiblingOrChild(error, V, NEXT)
    expect(model.FindMyNext(V, FIRST), precision)
    expect(error.FindPrevious(V), precision)

    # One predictive-coding step: p <- p + gain * (observation - p).
    observation = sensor.BuildAViewOverRegion(FF, F64)
    prediction = predictor.build_region_view(FF, F64)
    residual = error.BuildAViewOverRegion(ERROR, F64)
    gain = precision.BuildAViewOverRegion(WEIGHT, F64)
    assert all(view is not None and view.IsValid() for view in (
        observation, prediction, residual, gain
    ))
    assert observation.Size() == len(observation) == observation.size
    assert observation.GetProtocol() == observation.protocol == ac.RegionProtocol.ATOMIC_WORD_ARRAY
    assert gain.GetProtocol() == ac.RegionProtocol.PRIVATE_REGION

    assert observation.AtomicStore(0, 10.0)
    assert prediction.store(0, 6.0)                  # generic atomic dispatch
    gain[0] = 0.5                                    # PRIVATE_REGION dispatch
    error_value = observation.AtomicLoad(0) - prediction.load(0)
    assert residual.AtomicStore(0, error_value)
    desired = prediction.AtomicLoad(0) + gain[0] * residual[0]
    exchanged, observed = prediction.AtomicCompareExchangeStrong(0, 6.0, desired)
    assert exchanged and observed == 6.0 and prediction.AtomicLoad(0) == 8.0
    numpy_snapshot = prediction.to_numpy()           # safe copy, not slab memory
    assert numpy_snapshot[0] == 8.0

    # Schema mismatch, view zeroing, and APC-level zeroing.
    assert predictor.BuildAViewOverRegion(FF, ac.RegionDataType.UINT64_T) is None
    assert gain.fill_zero() and gain.load(0) == 0.0
    assert error.ZeroARegion(ERROR, F64) and residual.AtomicLoad(0) == 0.0

    # A live view pins only its APC. Retirement succeeds after that pin leaves scope.
    scratch_slot = scratch.get_this_slot_idx()
    pinned = scratch.BuildAViewOverRegion(FF, F64)
    assert pinned.AtomicStore(0, 99.0)
    assert not scratch.Retire()
    del pinned
    gc.collect()
    assert scratch.retire() and not scratch.is_valid()

    # The Fabric is full, so creation reclaims the retired slot with a new generation.
    replacement = make()
    assert replacement.GetThisSlotIdx() == scratch_slot
    assert replacement.IsActiveAPC() and not scratch.IsActiveAPC()

    fabric.ShutDownFabric()
    assert not fabric.IsFabricActive()
    assert not model.IsActiveAPC()
    fabric.ShutDownFabricWithPtrTable()               # idempotent public alias
    print("PUBLIC ARCHITECTURE MICRO-TEST: PASS (prediction 6 -> 8)")


if __name__ == "__main__":
    main()


PUBLIC ARCHITECTURE MICRO-TEST: PASS (prediction 6 -> 8)


In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

try:
    import atomiccim_bind as ac
except ImportError as exc:
    raise SystemExit(
        "atomiccim_bind is not importable. Build the Pybind11 module, then place "
        "it beside this script or add its directory to PYTHONPATH."
    ) from exc


def make_time_series():
    """Return a small noisy sine wave as two-lag supervised examples."""
    rng = np.random.default_rng(7)
    time = np.linspace(0.0, 12.0 * np.pi, 180)
    signal = np.sin(time) + rng.normal(0.0, 0.03, time.size)

    # Fit preprocessing on the training period only, avoiding test-data leakage.
    train_signal_count = 130
    scaler = StandardScaler().fit(signal[:train_signal_count, None])
    scaled = scaler.transform(signal[:, None]).ravel()

    features = np.array(
        [[scaled[t - 1], scaled[t - 2]] for t in range(2, scaled.size)],
        dtype=np.float32,
    )
    targets = scaled[2:].astype(np.float32)
    train_count = train_signal_count - 2
    return features[:train_count], targets[:train_count], features[train_count:], targets[train_count:]


def make_region_configuration():
    """Enable only the four float regions used by this experiment."""
    layout = ac.LayoutSpanAndPercentageCarrier()
    layout.FeedForward = 1
    layout.FeedBackward = 1
    layout.Lateral = 0
    layout.StateSlot = 0
    layout.ErrorSlot = 1
    layout.Weightless = 0
    layout.WeightSlot = 1
    layout.AUXSlot = 0
    layout.HeterogenousPtr = 0
    layout.FreeSlot = 0

    dtypes = ac.InitialRegionalDtypeConf()
    dtypes.ERROR_SLOT = ac.RegionDataType.FLOAT32_T
    dtypes.WEIGHT_SLOT = ac.RegionDataType.FLOAT32_T

    protocols = ac.InitialRegionalProtocol()
    protocols.ERROR_SLOT = ac.RegionProtocol.ATOMIC_WORD_ARRAY
    protocols.WEIGHT_SLOT = ac.RegionProtocol.ATOMIC_WORD_ARRAY
    return layout, dtypes, protocols


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def run_test():
    x_train, y_train, x_test, y_test = make_time_series()

    # scikit-learn does only the small learning step.
    learned_model = LinearRegression().fit(x_train, y_train)

    fabric = ac.Fabric()
    require(fabric.initialize(slot_count=8, slot_cell_count=256), "Fabric initialization failed")

    try:
        layout, dtypes, protocols = make_region_configuration()
        create = lambda **roots: fabric.create_apc(
            layout=layout, dtype=dtypes, protocol=protocols, **roots
        )

        predictor = create(wants_horizontal_root=True)
        sensor = create()
        error_unit = create()
        require(all((predictor, sensor, error_unit)), "APC creation failed")

        # H topology: PREDICTOR --first-child--> SENSOR --linked-child--> ERROR.
        require(
            sensor.attach_me_to_another(
                predictor, ac.Axis.HORIZONTAL, ac.Inheritance.FIRST_CHILD
            ),
            "Could not attach the sensor APC",
        )
        require(
            error_unit.attach_me_to_another(
                sensor, ac.Axis.HORIZONTAL, ac.Inheritance.LINKED_CHILD
            ),
            "Could not attach the error APC",
        )

        first = predictor.find_my_next(
            ac.Axis.HORIZONTAL, ac.Inheritance.FIRST_CHILD
        )
        require(first.found, "Predictor could not find its sensor child")
        second = first.apc.find_my_next(
            ac.Axis.HORIZONTAL, ac.Inheritance.LINKED_CHILD
        )
        require(second.found, "Sensor could not find its error sibling")
        require(first.apc.GetThisSlotIdx() == sensor.GetThisSlotIdx(), "Wrong first child")
        require(second.apc.GetThisSlotIdx() == error_unit.GetThisSlotIdx(), "Wrong sibling")

        column = ac.MacroColumnOfAPC
        dtype = ac.RegionDataType.FLOAT32_T
        observations = sensor.build_region_view(column.FEEDFORWARD_MESSAGE, dtype)
        predictions = predictor.build_region_view(column.FEEDBACKWARD_MESSAGE, dtype)
        errors = error_unit.build_region_view(column.ERROR_SLOT, dtype)
        weights = predictor.build_region_view(column.WEIGHT_SLOT, dtype)
        require(all((observations, predictions, errors, weights)), "A region view is unavailable")

        # The APC weight region is now the predictor's persistent parameter store.
        learned_values = [*learned_model.coef_, learned_model.intercept_]
        for index, value in enumerate(learned_values):
            require(weights.AtomicStore(index, float(value)), "Weight store failed")

        encoded_predictions = []
        encoded_errors = []
        for two_lags, actual in zip(x_test, y_test):
            require(observations.AtomicStore(0, float(two_lags[0])), "Input store failed")
            require(observations.AtomicStore(1, float(two_lags[1])), "Input store failed")
            require(observations.AtomicStore(2, float(actual)), "Target store failed")

            predicted = (
                weights.AtomicLoad(0) * observations.AtomicLoad(0)
                + weights.AtomicLoad(1) * observations.AtomicLoad(1)
                + weights.AtomicLoad(2)
            )
            residual = observations.AtomicLoad(2) - predicted

            require(predictions.AtomicStore(0, predicted), "Prediction store failed")
            require(errors.AtomicStore(0, residual), "Error store failed")
            encoded_predictions.append(predictions.AtomicLoad(0))
            encoded_errors.append(errors.AtomicLoad(0))

        encoded_predictions = np.asarray(encoded_predictions)
        encoded_errors = np.asarray(encoded_errors)
        reconstructed = encoded_predictions + encoded_errors

        predictive_mse = mean_squared_error(y_test, encoded_predictions)
        persistence_mse = mean_squared_error(y_test, x_test[:, 0])
        reconstruction_error = np.max(np.abs(y_test - reconstructed))

        require(predictive_mse < persistence_mse, "Predictor did not beat the naive baseline")
        require(reconstruction_error < 1e-5, "prediction + error did not reconstruct the input")

        improvement = 100.0 * (1.0 - predictive_mse / persistence_mse)
        print("APC SMALL PREDICTIVE-ENCODING TEST: PASS")
        print(
            "topology slots: "
            f"predictor {predictor.GetThisSlotIdx()} -> "
            f"sensor {sensor.GetThisSlotIdx()} -> error {error_unit.GetThisSlotIdx()}"
        )
        print(f"learned weights: {np.asarray(learned_values).round(4)}")
        print(f"predictive MSE: {predictive_mse:.6f}")
        print(f"naive MSE:      {persistence_mse:.6f}")
        print(f"improvement:    {improvement:.1f}%")
        print(f"decode error:   {reconstruction_error:.2e}")
    finally:
        fabric.shutdown()



In [2]:
if __name__ == "__main__":
    run_test()


APC SMALL PREDICTIVE-ENCODING TEST: PASS
topology slots: predictor 0 -> sensor 1 -> error 2
learned weights: [ 1.8581e+00 -9.0130e-01 -1.8000e-03]
predictive MSE: 0.009328
naive MSE:      0.049213
improvement:    81.0%
decode error:   5.96e-08
